<a href="https://www.kaggle.com/code/adithyan65/malayalamcybercon-data" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# MalayalamCyberCon — Multi-Model Training Notebook
### Conflict & Cyberbullying Detection in Manglish YouTube Comments

---

## Project Overview

This notebook trains and compares **three transformer models** on a multi-task cyberbullying detection pipeline for Malayalam-English code-mixed (Manglish) YouTube comments.

**Pipeline:** Each comment thread goes through sequential classifiers:
1. **Conflict Detection** (binary) — Is there cyberbullying?
2. **Severity Classification** (3-class) — How bad is it?
3. **Type Classification** (4-class) — What kind of bullying?
4. **Target Classification** (3-class) — Who is being targeted?

Tasks 2–4 only apply to comments flagged as conflict=1 by Task 1.

---

## Before Running

| Step | Action |
|------|--------|
| **1. Accelerator** | Settings → Accelerator → **GPU T4 x2** |
| **2. Dataset** | Add Data → `malayalamcybercon-dataset1` (train/val/test CSVs) |
| **3. Run** | Run All — takes 30–40 min per model (~2 hours total) |

---

## Label Schema (0-indexed)

| Task | Labels | Notes |
|------|--------|-------|
| `label_conflict` | 0 = no conflict, 1 = conflict | Binary — trained on all rows |
| `label_severity` | 0 = mild, 1 = moderate, 2 = severe | conflict=1 rows only |
| `label_type` | 0 = personal, 1 = political, 2 = sexual/gendered, 3 = threat | conflict=1 rows only |
| `label_target` | 0 = commenter, 1 = creator/public figure, 2 = community/group | conflict=1 rows only |

---

## Models Compared

| Model | Parameters | Why? |
|-------|-----------|------|
| `xlm-roberta-base` | 278M | Strong multilingual baseline, 100 languages |
| `google/muril-base-cased` | 237M | Pretrained on 17 Indian languages incl. transliterated Malayalam |
| `xlm-roberta-large` | 560M | Larger capacity, potential upper bound on performance |


## 1 — Install Dependencies

Install HuggingFace Transformers, Accelerate (for mixed-precision training), and scikit-learn (for metrics and class weighting). The `%%capture` magic suppresses the noisy pip output.

In [1]:
%%capture
!pip install transformers accelerate scikit-learn

## 2 — Configuration

All hyperparameters are centralised here so you can tweak them without touching any other cell.

**Key design decisions:**
- **Task-specific learning rates:** Severity/type/target use a lower LR (1e-5) than conflict (2e-5) because they train on smaller subsets and overfit faster.
- **More epochs for sub-tasks:** Conflict has the most data → 5 epochs is enough. Severity/type/target have fewer rows → they need 10–15 epochs to converge.
- **Batch size override for XLM-R Large:** At 560M params, it does not fit in T4 VRAM with batch=16, so we drop to 8.

In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────────

DATA_DIR    = '/kaggle/input/datasets/adithyan65/malayalamcybercon-dataset1'
OUTPUT_DIR  = '/kaggle/working/models'
MAX_LEN     = 256        # Max token length per input (covers 99%+ of comments)
SEED        = 42         # Reproducibility seed
TASKS       = ['conflict', 'severity', 'type', 'target']

# Task-specific hyperparameters
LR_BY_TASK         = {'conflict': 2e-5,  'severity': 1e-5,  'type': 1e-5,  'target': 1e-5}
EPOCHS             = {'conflict': 5,     'severity': 10,    'type': 15,    'target': 10}
BATCH_SIZE_DEFAULT = 16

# Models to train and compare
MODELS_TO_COMPARE = [
    ('xlm-roberta-base',        'xlmr_base'),
    ('google/muril-base-cased', 'muril_base'),
    ('xlm-roberta-large',       'xlmr_large'),
]

# XLM-RoBERTa-large needs a smaller batch to fit in T4 GPU VRAM (16 GB)
BATCH_SIZE_OVERRIDE = {
    'xlmr_large': 8,
}

print("Configuration loaded.")
print(f"  Tasks:  {TASKS}")
print(f"  Models: {[m[0] for m in MODELS_TO_COMPARE]}")

Configuration loaded.
  Tasks:  ['conflict', 'severity', 'type', 'target']
  Models: ['xlm-roberta-base', 'google/muril-base-cased', 'xlm-roberta-large']


## 3 — Imports & Device Check

We import everything up front so missing packages fail early, not 30 minutes into training.

In [3]:
# ── Imports ─────────────────────────────────────────────────────────────────────
import copy, csv, random as _random
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score, classification_report
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    Trainer, TrainingArguments, TrainerCallback, set_seed,
)

set_seed(SEED)

# ── Device check ─────────────────────────────────────────────────────────────────
device = 'GPU' if torch.cuda.is_available() else 'CPU'
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected! Training will be extremely slow.')
    print('         Go to Settings > Accelerator > GPU T4 x2')

Device : GPU
GPU    : Tesla T4
VRAM   : 15.6 GB


## 4 — Disk Cleanup

Kaggle gives you ~20 GB of working disk. Old checkpoints from previous runs can eat this up fast, causing cryptic "No space left on device" errors mid-training. We wipe the working directory at the start to guarantee a clean slate.

In [4]:
# ── Disk cleanup ──────────────────────────────────────────────────────────────────
import shutil

for d in Path('/kaggle/working').iterdir():
    try:
        shutil.rmtree(d) if d.is_dir() else d.unlink()
    except:
        pass

_, used, free = shutil.disk_usage('/kaggle/working')
print(f'Disk: {free/1e9:.1f} GB free ({used/1e9:.1f} GB used)')

Disk: 20.9 GB free (0.0 GB used)


## 5 — Load Data Splits

The dataset has pre-made train/val/test CSV splits. Each row contains:
- `thread_text` — the concatenated comment thread (parent + replies)
- `label_conflict` — 0 or 1
- `label_severity` — 0/1/2 (only meaningful when conflict=1)
- `label_type` — 0/1/2/3 (only meaningful when conflict=1)
- `label_target` — 0/1/2 (only meaningful when conflict=1)

**Important:** Rows where conflict=1 but severity is missing are skipped — they are annotation errors in the original data.

In [5]:
# ── Load pre-split CSV files ───────────────────────────────────────────────────
def load_split(path):
    """Load a train/val/test CSV split, parsing all four label columns."""
    rows = []
    skipped = 0
    with open(path, encoding='utf-8-sig') as f:
        for row in csv.DictReader(f):
            lc = row.get('label_conflict', '').strip()
            ls = row.get('label_severity', '').strip()
            lt = row.get('label_type', '').strip()
            lg = row.get('label_target', '').strip()   # target label

            if not lc:
                continue

            lc_int = int(lc)

            # Skip conflict=1 rows that are missing severity (annotation error)
            if lc_int == 1 and not ls:
                skipped += 1
                continue

            ls_int = int(ls) if ls else 0
            lt_int = int(lt) if lt else 0
            lg_int = (int(lg) - 1) if lg else 0   # CSV is 1-indexed → convert to 0-indexed
            rows.append({
                'text':           row['thread_text'],
                'label_conflict': lc_int,
                'label_severity': ls_int,
                'label_type':     lt_int,
                'label_target':   lg_int,
            })
    if skipped:
        print(f'  [WARN] {Path(path).name}: skipped {skipped} rows (conflict=1 but missing severity)')
    return rows


# ── Load all splits ────────────────────────────────────────────────────────────
train_all = load_split(f'{DATA_DIR}/train.csv')
val_all   = load_split(f'{DATA_DIR}/val.csv')
test_all  = load_split(f'{DATA_DIR}/test.csv')

print(f'{"Split":<8} {"Total":>6} {"Conflict=1":>12} {"% Conflict":>12}')
print("-" * 42)
for name, split in [('train', train_all), ('val', val_all), ('test', test_all)]:
    c1 = sum(1 for r in split if r['label_conflict'] == 1)
    print(f'{name:<8} {len(split):>6} {c1:>12} {100*c1/len(split):>11.1f}%')

# ── Show class distributions for sub-tasks ───────────────────────────────────────
print('\n--- Class distributions (conflict=1 rows in train split) ---')
conflict_rows = [r for r in train_all if r['label_conflict'] == 1]
for task in ['severity', 'type', 'target']:
    col = f'label_{task}'
    counts = Counter(r[col] for r in conflict_rows)
    total = sum(counts.values())
    dist = '  '.join(f'{k}:{v}({100*v/total:.1f}%)' for k, v in sorted(counts.items()))
    print(f'  {task:<10}: {dist}')

Split     Total   Conflict=1   % Conflict
------------------------------------------
train      1168          389        33.3%
val         249           83        33.3%
test        252           84        33.3%

--- Class distributions (conflict=1 rows in train split) ---
  severity  : 0:158(40.6%)  1:147(37.8%)  2:84(21.6%)
  type      : 0:261(67.1%)  1:58(14.9%)  2:60(15.4%)  3:10(2.6%)
  target    : 0:186(47.8%)  1:141(36.2%)  2:62(15.9%)


## 5b — Validate & Fix Label Ranges

The model expects **0-indexed labels** (e.g., severity: 0, 1, 2). Some CSV datasets use **1-indexed labels** (1, 2, 3) which causes a silent CUDA crash with a cryptic `device-side assert triggered` error.

This cell checks every label column, reports what it found, and auto-converts 1-indexed labels to 0-indexed if needed.

In [6]:
# ── Validate and fix label ranges ─────────────────────────────────────────────
EXPECTED_RANGES = {
    'label_conflict': (0, 1),    # 2 classes: 0, 1
    'label_severity': (0, 2),    # 3 classes: 0, 1, 2
    'label_type':     (0, 3),    # 4 classes: 0, 1, 2, 3
    'label_target':   (0, 2),    # 3 classes: 0, 1, 2
}

def validate_and_fix_labels(splits, split_names):
    """Check label ranges and auto-convert 1-indexed → 0-indexed if needed."""
    for col, (exp_min, exp_max) in EXPECTED_RANGES.items():
        # Gather all values across splits
        all_vals = []
        for split in splits:
            all_vals.extend(r[col] for r in split)
        
        if not all_vals:
            print(f"  {col}: NO DATA")
            continue
        
        actual_min = min(all_vals)
        actual_max = max(all_vals)
        unique_vals = sorted(set(all_vals))
        
        if actual_min == exp_min and actual_max <= exp_max:
            print(f"  {col}: OK  (range [{actual_min}, {actual_max}], unique={unique_vals})")
        elif actual_min == exp_min + 1 and actual_max == exp_max + 1:
            # 1-indexed! Auto-fix by subtracting 1
            print(f"  {col}: 1-INDEXED detected (range [{actual_min}, {actual_max}]) → converting to 0-indexed")
            for split in splits:
                for r in split:
                    r[col] -= 1
            new_unique = sorted(set(r[col] for split in splits for r in split))
            print(f"           Fixed! Now: {new_unique}")
        elif actual_max > exp_max:
            print(f"  {col}: WARNING — values go up to {actual_max}, expected max {exp_max}")
            print(f"           unique={unique_vals}")
            print(f"           This WILL crash during training!")
        else:
            print(f"  {col}: range [{actual_min}, {actual_max}], unique={unique_vals}")

print("Label validation (all splits combined):")
print("-" * 60)
validate_and_fix_labels(
    [train_all, val_all, test_all],
    ['train', 'val', 'test']
)

# Also re-check conflict=1 sub-task distributions after any fixes
print("\n--- Class distributions after validation (conflict=1, train) ---")
conflict_rows = [r for r in train_all if r['label_conflict'] == 1]
for task in ['severity', 'type', 'target']:
    col = f'label_{task}'
    counts = Counter(r[col] for r in conflict_rows)
    total = sum(counts.values())
    dist = '  '.join(f'{k}:{v}({100*v/total:.1f}%)' for k, v in sorted(counts.items()))
    print(f'  {task:<10}: {dist}')


Label validation (all splits combined):
------------------------------------------------------------
  label_conflict: OK  (range [0, 1], unique=[0, 1])
  label_severity: OK  (range [0, 2], unique=[0, 1, 2])
  label_type: OK  (range [0, 3], unique=[0, 1, 2, 3])
  label_target: OK  (range [0, 2], unique=[0, 1, 2])

--- Class distributions after validation (conflict=1, train) ---
  severity  : 0:158(40.6%)  1:147(37.8%)  2:84(21.6%)
  type      : 0:261(67.1%)  1:58(14.9%)  2:60(15.4%)  3:10(2.6%)
  target    : 0:186(47.8%)  1:141(36.2%)  2:62(15.9%)


## 6 — Model Components

This section defines the core building blocks:

### 6.1 — Oversampling
For highly imbalanced tasks (type, target), we duplicate minority-class rows until every class has the same count as the majority class. Simple but effective.

### 6.2 — ThreadDataset
A standard PyTorch Dataset that tokenises comment threads and pairs them with labels.

### 6.3 — FocalLossTrainer
A custom HuggingFace Trainer that replaces the default cross-entropy loss:

- **For conflict/type/target tasks:** Uses **Focal Loss**. Imagine a classroom where 97% of students already understand the lesson. Standard teaching (cross-entropy) spends equal time on everyone. Focal loss says "ignore the kids who already get it, spend all your energy on the 3% who are struggling." The `gamma` parameter controls how aggressively it ignores easy examples (gamma=0 is standard CE, gamma=2 strongly focuses on hard cases).

- **For severity task:** Uses **Ordinal Soft-Label Loss**. Severity is ordered (mild < moderate < severe), but standard classification treats all mistakes equally — predicting "mild" when the truth is "severe" gets the same penalty as predicting "moderate." Ordinal loss fixes this by bleeding 5% probability to adjacent classes, so the model learns that mild→severe is a bigger mistake than mild→moderate.

### 6.4 — BestModelInMemory
A callback that watches validation F1 after each epoch and keeps a copy of the best model weights in CPU RAM. This avoids writing checkpoints to disk (saving ~1 GB per checkpoint × 3 models × 4 tasks = up to 12 GB of disk savings).

In [7]:
# ── Oversampling utility ──────────────────────────────────────────────────────────
def oversample_to_balance(rows, label_col, seed=42):
    """Duplicate minority-class rows so every class matches the majority count."""
    rng = _random.Random(seed)
    counts = Counter(r[label_col] for r in rows)
    max_count = max(counts.values())
    balanced = list(rows)
    for cls, cnt in counts.items():
        deficit = max_count - cnt
        if deficit > 0:
            cls_rows = [r for r in rows if r[label_col] == cls]
            balanced.extend(rng.choices(cls_rows, k=deficit))
    rng.shuffle(balanced)
    return balanced


# ── Dataset ───────────────────────────────────────────────────────────────────────
class ThreadDataset(Dataset):
    """Tokenises comment threads and pairs them with integer labels."""
    def __init__(self, rows, tokenizer, max_len, label_col):
        self.labels    = [r[label_col] for r in rows]
        self.encodings = tokenizer(
            [r['text'] for r in rows],
            truncation=True, padding='max_length',
            max_length=max_len, return_tensors='pt',
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


# ── Focal Loss Trainer ────────────────────────────────────────────────────────────
class FocalLossTrainer(Trainer):
    """
    Custom trainer with task-specific loss functions:
      - conflict/type/target : Focal loss with class weights
      - severity             : Ordinal soft-label loss
    """
    def __init__(self, *args, class_weights=None, gamma=2.0, task='conflict', **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.gamma = gamma
        self.task  = task

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits
        w = self.class_weights.to(logits.device) if self.class_weights is not None else None

        if self.task == 'severity':
            # Ordinal soft-label loss
            num_classes = logits.size(-1)
            soft = F.one_hot(labels, num_classes).float()
            transfer = 0.05
            for i in range(len(labels)):
                lbl = labels[i].item()
                if lbl > 0:
                    soft[i, lbl]     -= transfer
                    soft[i, lbl - 1] += transfer
                if lbl < num_classes - 1:
                    soft[i, lbl]     -= transfer
                    soft[i, lbl + 1] += transfer
            log_probs  = F.log_softmax(logits, dim=-1)
            loss_per_s = -(soft * log_probs).sum(dim=-1)
            if w is not None:
                loss_per_s = loss_per_s * w[labels]
            loss = loss_per_s.mean()
        else:
            # Focal loss
            ce   = F.cross_entropy(logits, labels, weight=w, reduction='none')
            pt   = torch.exp(-ce)
            loss = ((1 - pt) ** self.gamma * ce).mean()

        return (loss, outputs) if return_outputs else loss


# ── Best-model-in-memory callback ─────────────────────────────────────────────────
class BestModelInMemory(TrainerCallback):
    """
    Watches validation F1 after each epoch and keeps the best model weights
    in CPU RAM. Avoids writing checkpoint files to disk.
    """
    def __init__(self):
        self.best_f1    = -1.0
        self.best_state = None

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        model = kwargs.get('model')
        if model is None:
            return
        f1 = (metrics or {}).get('eval_f1_macro', -1.0)
        if f1 > self.best_f1:
            self.best_f1    = f1
            self.best_state = {k: v.detach().cpu().clone()
                               for k, v in model.state_dict().items()}
            print(f'  * New best f1_macro: {f1:.4f} — saved in memory')

print("Model components defined: ThreadDataset, FocalLossTrainer, BestModelInMemory")

Model components defined: ThreadDataset, FocalLossTrainer, BestModelInMemory


## 7 — Metrics & Reporting

Two helper functions:
- `make_compute_metrics(task)` — Returns a function that HuggingFace Trainer calls after each epoch to compute F1 and accuracy on the validation set. For severity, it also tracks MAE (mean absolute error) on the ordinal scale.
- `print_report(task, y_true, y_pred)` — Prints a detailed sklearn classification report with human-readable class names.

In [8]:
# ── Metrics ───────────────────────────────────────────────────────────────────────
def make_compute_metrics(task):
    """Returns a metric function that Trainer calls after each epoch."""
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds    = np.argmax(logits, axis=-1)
        f1_macro = f1_score(labels, preds, average='macro', zero_division=0)
        acc      = accuracy_score(labels, preds)
        result   = {'f1_macro': f1_macro, 'accuracy': acc}
        if task == 'severity':
            result['mae'] = float(np.mean(np.abs(labels - preds)))
        return result
    return compute_metrics


# ── Classification report with human-readable names ──────────────────────────────
LABEL_NAMES = {
    'conflict': {
        'labels': [0, 1],
        'names':  ['no_conflict', 'conflict'],
    },
    'severity': {
        'labels': [0, 1, 2],
        'names':  ['mild', 'moderate', 'severe'],
    },
    'type': {
        'labels': [0, 1, 2, 3],
        'names':  ['personal', 'political', 'sexual/gendered', 'threat'],
    },
    'target': {
        'labels': [0, 1, 2],
        'names':  ['commenter', 'creator/pub_fig', 'community/group'],
    },
}


def print_report(task, y_true, y_pred):
    """Print a formatted classification report for the given task."""
    info = LABEL_NAMES[task]
    print(classification_report(
        y_true, y_pred,
        labels=info['labels'],
        target_names=info['names'],
        zero_division=0
    ))

print("Metric functions defined.")

Metric functions defined.


## 8 — Training Function

This is the main workhorse. For each (model, task) combination, it:

1. **Filters rows** — conflict uses all rows; severity/type/target use only conflict=1 rows
2. **Oversamples minority classes** — for type and target tasks (e.g., "threat" is only ~3% of conflict rows)
3. **Computes class weights** — balanced weights inversely proportional to class frequency
4. **Tokenises** with the specific model tokenizer
5. **Trains** with focal loss (or ordinal loss for severity)
6. **Restores best checkpoint** from memory after training
7. **Evaluates on test set** and prints a full classification report
8. **Saves the best model** to disk and frees GPU memory

In [9]:
# ── Train one task for one model ──────────────────────────────────────────────────
def train_task(task, train_all, val_all, test_all, model_id, model_slug):
    """Train a single classifier for one task using one model.
    Returns (save_path, macro_f1_on_test)."""
    from sklearn.metrics import f1_score as sk_f1
    
    label_col  = f'label_{task}'
    if task == 'conflict':
        num_labels = 2
    elif task in ('severity', 'target'):
        num_labels = 3
    else:  # type
        num_labels = 4
    batch_size = BATCH_SIZE_OVERRIDE.get(model_slug, BATCH_SIZE_DEFAULT)

    # Filter rows
    if task == 'conflict':
        train_rows, val_rows, test_rows = train_all, val_all, test_all
    else:
        train_rows = [r for r in train_all if r['label_conflict'] == 1]
        val_rows   = [r for r in val_all   if r['label_conflict'] == 1]
        test_rows  = [r for r in test_all  if r['label_conflict'] == 1]

    print(f'\n{"─"*60}')
    print(f'  Task: {task.upper()}  |  num_labels={num_labels}  |  '
          f'train:{len(train_rows)}  val:{len(val_rows)}  test:{len(test_rows)}')
    print(f'{"─"*60}')

    if len(train_rows) < 20:
        print('  [SKIP] Not enough samples.')
        return None, 0.0

    # Oversample for type/target
    if task in ('type', 'target'):
        train_rows = oversample_to_balance(train_rows, label_col, seed=SEED)
        oc = Counter(r[label_col] for r in train_rows)
        print(f'  After oversampling: {len(train_rows)} rows — ' +
              '  '.join(f'class {c}:{n}' for c, n in sorted(oc.items())))

    # Compute balanced class weights
    arr     = np.array([r[label_col] for r in train_rows])
    classes = np.unique(arr)
    weights = compute_class_weight('balanced', classes=classes, y=arr)
    class_weights = torch.tensor(weights, dtype=torch.float)
    print('  Class weights: ' + '  '.join(f'class {c}->{w:.3f}' for c, w in zip(classes, weights)))

    # Task-specific hyperparameters
    gamma = 0.0 if task == 'conflict' else (1.0 if task in ('severity', 'target') else 2.0)
    lr    = LR_BY_TASK[task]
    print(f'  Focal gamma: {gamma}  |  LR: {lr}  |  Batch: {batch_size}')

    # Safety check: ensure labels are in valid range
    label_vals = set(r[label_col] for r in train_rows)
    if max(label_vals) >= num_labels or min(label_vals) < 0:
        raise ValueError(
            f"Label mismatch for {task}! Labels {sorted(label_vals)} but "
            f"num_labels={num_labels}. Expected range [0, {num_labels-1}]. "
            f"Check if your CSV uses 1-indexed labels."
        )

    # Tokenise
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    train_ds  = ThreadDataset(train_rows, tokenizer, MAX_LEN, label_col)
    val_ds    = ThreadDataset(val_rows,   tokenizer, MAX_LEN, label_col)
    test_ds   = ThreadDataset(test_rows,  tokenizer, MAX_LEN, label_col)

    # Load model
    model    = AutoModelForSequenceClassification.from_pretrained(
                   model_id, num_labels=num_labels, ignore_mismatched_sizes=True)
    task_out = Path(OUTPUT_DIR) / model_slug / task
    task_out.mkdir(parents=True, exist_ok=True)

    # Training arguments
    training_args = TrainingArguments(
        output_dir                  = str(task_out),
        num_train_epochs            = EPOCHS[task],
        per_device_train_batch_size = batch_size,
        per_device_eval_batch_size  = batch_size,
        learning_rate               = lr,
        warmup_ratio                = 0.15,
        weight_decay                = 0.01,
        eval_strategy               = 'epoch',
        save_strategy               = 'no',
        load_best_model_at_end      = False,
        logging_steps               = 20,
        seed                        = SEED,
        report_to                   = 'none',
        fp16                        = torch.cuda.is_available(),
    )

    # Train
    best_cb = BestModelInMemory()
    trainer = FocalLossTrainer(
        model           = model,
        args            = training_args,
        train_dataset   = train_ds,
        eval_dataset    = val_ds,
        compute_metrics = make_compute_metrics(task),
        callbacks       = [best_cb],
        class_weights   = class_weights,
        gamma           = gamma,
        task            = task,
    )

    print(f'  Training on {device}...')
    trainer.train()

    # Restore best model from memory
    if best_cb.best_state is not None:
        model.load_state_dict({k: v.to(model.device) for k, v in best_cb.best_state.items()})
        print(f'  Loaded best model (f1={best_cb.best_f1:.4f}) from memory')
    else:
        print('  WARNING: best_state is None — using last epoch model')

    # Evaluate on test set
    preds_out = trainer.predict(test_ds)
    y_pred    = np.argmax(preds_out.predictions, axis=-1).tolist()
    y_true    = [r[label_col] for r in test_rows]
    print(f'\n  Test set report ({task}):')
    print_report(task, y_true, y_pred)
    macro_f1 = sk_f1(y_true, y_pred, average='macro', zero_division=0)

    # Save best model
    best_dir = task_out / 'best'
    best_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(best_dir))
    tokenizer.save_pretrained(str(best_dir))
    print(f'  Saved -> {best_dir}')

    # Free GPU memory
    del model, trainer
    torch.cuda.empty_cache()

    return best_dir, macro_f1

print("Training function defined.")

Training function defined.


## 9 — Run Training (All Models × All Tasks)

This is where the actual training happens. For each of the 3 models, we train all 4 tasks sequentially. After everything finishes, a comparison table shows macro F1 scores side by side.

**Expected runtime:** ~30–40 min per model on a T4 GPU, so roughly **2 hours total** for all 3 models.

In [10]:
# ── Run all models x all tasks ────────────────────────────────────────────────────
results = {}  # {model_slug: {task: macro_f1}}

for model_id, model_slug in MODELS_TO_COMPARE:
    print(f'\n{"="*60}')
    print(f'  MODEL: {model_id}  ({model_slug})')
    print(f'{"="*60}')
    results[model_slug] = {}
    for task in TASKS:
        try:
            _, f1 = train_task(task, train_all, val_all, test_all, model_id, model_slug)
            results[model_slug][task] = round(f1, 4)
        except Exception as e:
            print(f'  [ERROR] {task}: {e}')
            import traceback; traceback.print_exc()
            results[model_slug][task] = None

# ── Comparison table ───────────────────────────────────────────────────────────────
print('\n\n' + '='*80)
print('  COMPARISON TABLE — Macro F1 on Test Set')
print('='*80)
header = f'  {"Model":<28} {"Conflict":>10} {"Severity":>10} {"Type":>10} {"Target":>10}'
print(header)
print('  ' + '─'*68)
for model_id, model_slug in MODELS_TO_COMPARE:
    r = results.get(model_slug, {})
    def fmt(v): return f'{v:.4f}' if v is not None else '   N/A'
    print(f'  {model_id:<28} {fmt(r.get("conflict")):>10} {fmt(r.get("severity")):>10} '
          f'{fmt(r.get("type")):>10} {fmt(r.get("target")):>10}')
print('='*80)


  MODEL: xlm-roberta-base  (xlmr_base)

────────────────────────────────────────────────────────────
  Task: CONFLICT  |  num_labels=2  |  train:1168  val:249  test:252
────────────────────────────────────────────────────────────
  Class weights: class 0->0.750  class 1->1.501
  Focal gamma: 0.0  |  LR: 2e-05  |  Batch: 16


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,0.771911,0.754265,0.181287,0.373494
2,0.748959,0.772936,0.090278,0.156627
3,0.746942,0.766470,0.253247,0.361446
4,0.697932,0.701197,0.368742,0.409639
5,0.627855,0.641701,0.406649,0.506024
6,0.437579,0.812336,0.361361,0.445783
7,0.299062,0.809350,0.452452,0.530120
8,0.240511,0.853730,0.367414,0.457831
9,0.197275,0.935854,0.423674,0.530120
10,0.177081,0.915237,0.385521,0.481928


  * New best f1_macro: 0.1813 — saved in memory
  * New best f1_macro: 0.2532 — saved in memory
  * New best f1_macro: 0.3687 — saved in memory
  * New best f1_macro: 0.4066 — saved in memory
  * New best f1_macro: 0.4525 — saved in memory
  Loaded best model (f1=0.4525) from memory



  Test set report (target):
                 precision    recall  f1-score   support

      commenter       0.57      0.46      0.51        46
creator/pub_fig       0.38      0.54      0.44        28
community/group       0.43      0.30      0.35        10

       accuracy                           0.46        84
      macro avg       0.46      0.43      0.43        84
   weighted avg       0.49      0.46      0.47        84



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Saved -> /kaggle/working/models/xlmr_large/target/best


  COMPARISON TABLE — Macro F1 on Test Set
  Model                          Conflict   Severity       Type     Target
  ────────────────────────────────────────────────────────────────────
  xlm-roberta-base                 0.7529     0.3888     0.3444     0.3757
  google/muril-base-cased          0.6780     0.3682     0.3370     0.2485
  xlm-roberta-large                0.7580     0.4111     0.3094     0.4334


## 10 — Inference Demo

Run the trained models on sample Manglish comments to verify they work. This uses the best-performing model (XLM-R large) saved in the previous step.

The pipeline runs sequentially:
1. **Conflict** → Is this cyberbullying? (runs on every input)
2. **Severity** → How bad? (only if conflict=1)
3. **Type** → What kind? (only if conflict=1)
4. **Target** → Who is targeted? (only if conflict=1)

In [11]:
# ── Inference Demo ────────────────────────────────────────────────────────────────

DEMO_MODEL = 'xlmr_large'   # Change to 'xlmr_base' or 'muril_base' if needed
DEMO_DIR   = f'{OUTPUT_DIR}/{DEMO_MODEL}'

LABELS = {
    'conflict': ['No Conflict', 'Conflict'],
    'severity': ['Mild', 'Moderate', 'Severe'],
    'type':     ['Personal', 'Political', 'Sexual/Gendered', 'Threat'],
    'target':   ['Commenter', 'Creator/Public Figure', 'Community/Group'],
}

def load_task_model(task):
    task_path = f'{DEMO_DIR}/{task}/best'
    tok = AutoTokenizer.from_pretrained(task_path)
    mdl = AutoModelForSequenceClassification.from_pretrained(task_path)
    mdl.eval()
    if torch.cuda.is_available():
        mdl.cuda()
    return tok, mdl

def predict_one(text, tokenizer, model):
    inputs = tokenizer(text, truncation=True, padding='max_length',
                       max_length=MAX_LEN, return_tensors='pt')
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim=-1).squeeze().cpu()
    return probs.argmax().item(), probs.tolist()

def run_demo(text):
    print(f'\nINPUT: "{text}"')
    print("-" * 50)

    tok, mdl = load_task_model('conflict')
    pred, probs = predict_one(text, tok, mdl)
    print(f'  Conflict : {LABELS["conflict"][pred]:20s}  ({" | ".join(f"{LABELS["conflict"][i]}={p:.3f}" for i, p in enumerate(probs))})')
    del tok, mdl

    if pred == 0:
        print(f'  >> No conflict detected. Sub-tasks skipped.')
        return

    for task in ['severity', 'type', 'target']:
        tok, mdl = load_task_model(task)
        pred, probs = predict_one(text, tok, mdl)
        print(f'  {task.capitalize():9s}: {LABELS[task][pred]:20s}  ({" | ".join(f"{LABELS[task][i]}={p:.3f}" for i, p in enumerate(probs))})')
        del tok, mdl

    torch.cuda.empty_cache()


# ── Test cases ───────────────────────────────────────────────────────────────────
test_texts = [
    # Expected: No conflict (positive comment)
    "[1] ee video kollam, nalla content [2\u2605] adipoli explanation bro",
    # Expected: Conflict (mild, personal)
    "[1] nee onnum ariyilla mone [2\u2605] pani thudangeda",
    # Expected: Conflict (threatening)
    "[1] ninte veedu ariyam enikku [2\u2605] irangiyaal kanum",
    # Expected: No conflict (question)
    "[1] ee app evide download cheyyum? [2\u2605] link tharumo",
]

print(f"Running inference demo with {DEMO_MODEL}...")
print("=" * 50)
for text in test_texts:
    run_demo(text)
print("\n" + "=" * 50)
print("Demo complete!")

Running inference demo with xlmr_large...

INPUT: "[1] ee video kollam, nalla content [2★] adipoli explanation bro"
--------------------------------------------------


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

  Conflict : No Conflict           (No Conflict=0.990 | Conflict=0.010)
  >> No conflict detected. Sub-tasks skipped.

INPUT: "[1] nee onnum ariyilla mone [2★] pani thudangeda"
--------------------------------------------------


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

  Conflict : No Conflict           (No Conflict=0.584 | Conflict=0.416)
  >> No conflict detected. Sub-tasks skipped.

INPUT: "[1] ninte veedu ariyam enikku [2★] irangiyaal kanum"
--------------------------------------------------


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

  Conflict : No Conflict           (No Conflict=0.666 | Conflict=0.334)
  >> No conflict detected. Sub-tasks skipped.

INPUT: "[1] ee app evide download cheyyum? [2★] link tharumo"
--------------------------------------------------


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

  Conflict : No Conflict           (No Conflict=0.991 | Conflict=0.009)
  >> No conflict detected. Sub-tasks skipped.

Demo complete!


## 11 — Zip & Download Models

After training, we zip each model for download. Each zip contains one folder per task (conflict/severity/type/target), each with the saved model weights, config, and tokenizer files.

In [12]:
# # ── Clean up disk, then zip models ───────────────────────────────────────────
# import shutil

# # Remove old zips and junk, keep models folder
# for item in Path('/kaggle/working').iterdir():
#     if item.name == 'models':
#         continue
#     try:
#         shutil.rmtree(item) if item.is_dir() else item.unlink()
#     except:
#         pass

# _, used, free = shutil.disk_usage('/kaggle/working')
# print(f'After cleanup: {free/1e9:.1f} GB free')

# # Only zip the best model to save disk space
# TO_ZIP = ['xlmr_large']

# for model_slug in TO_ZIP:
#     model_dir = Path(OUTPUT_DIR) / model_slug
#     if not model_dir.exists():
#         print(f'  {model_slug}: not found, skipping')
#         continue
#     zip_path = f'/kaggle/working/{model_slug}'
#     shutil.make_archive(zip_path, 'zip', OUTPUT_DIR, model_slug)
#     size_gb = Path(zip_path + '.zip').stat().st_size / 1e9
#     _, used, free = shutil.disk_usage('/kaggle/working')
#     print(f'  {model_slug}.zip  ->  {size_gb:.2f} GB  (disk free: {free/1e9:.1f} GB)')

# print('\nDone! Download from the Output tab.')

---

## After Downloading

### 1. Verify models are trained (not random)
```python
from safetensors import safe_open
for task in ['conflict', 'severity', 'type', 'target']:
    with safe_open(f'models/{task}/best/model.safetensors', framework='pt', device='cpu') as f:
        w = f.get_tensor('classifier.weight')
        print(f'{task}: mean_abs={w.abs().mean():.4f}')
```
If `mean_abs` is near 0.01–0.05, the model trained. If it is near 0.0001, something went wrong.

### 2. Run inference
```bash
python src/predict.py --text "[1] ee video kollam, nalla content [2★] adipoli explanation bro"
```

### Expected Results (approximate)

| Task | Metric | Target |
|------|--------|--------|
| Conflict | macro F1 | ≥ 0.77 |
| Severity | macro F1 | ≥ 0.40 |
| Type | macro F1 | ≥ 0.30 |
| Target | macro F1 | ≥ 0.35 |

---

*Notebook of MalayalamCyberCon Project, Amrita Vishwa Vidyapeetham*

In [13]:
# """
# MalayalamCyberCon — Inference Script
# Runs the 4-task cyberbullying detection pipeline on Manglish text.

# Usage:
#     python predict.py --text "nee oru veliya poda myre"
#     python predict.py --text "ee video kollam, nalla content"
#     python predict.py --model_dir ./xlmr_large

# Requirements:
#     pip install transformers torch safetensors
# """

# import argparse
# import torch
# from pathlib import Path
# from transformers import AutoTokenizer, AutoModelForSequenceClassification


# # ── Label mappings ───────────────────────────────────────────────────────────
# LABELS = {
#     'conflict': ['No Conflict', 'Conflict'],
#     'severity': ['Mild', 'Moderate', 'Severe'],
#     'type':     ['Personal', 'Political', 'Sexual/Gendered', 'Threat'],
#     'target':   ['Commenter', 'Creator/Public Figure', 'Community/Group'],
# }

# TASKS = ['conflict', 'severity', 'type', 'target']


# def load_model(model_dir, task):
#     """Load a trained model and tokenizer for a specific task."""
#     task_path = Path(model_dir) / task / 'best'
#     if not task_path.exists():
#         raise FileNotFoundError(f"Model not found at {task_path}")

#     tokenizer = AutoTokenizer.from_pretrained(str(task_path))
#     model = AutoModelForSequenceClassification.from_pretrained(str(task_path))
#     model.eval()
#     return tokenizer, model


# def predict_task(text, tokenizer, model):
#     """Run inference for a single task. Returns (predicted_class, probabilities)."""
#     inputs = tokenizer(
#         text, truncation=True, padding='max_length',
#         max_length=256, return_tensors='pt'
#     )

#     with torch.no_grad():
#         outputs = model(**inputs)
#         probs = torch.softmax(outputs.logits, dim=-1).squeeze()

#     pred_class = probs.argmax().item()
#     return pred_class, probs.tolist()


# def run_pipeline(text, model_dir):
#     """Run the full 4-task pipeline on a single text input."""

#     print(f"\n{'='*60}")
#     print(f"  INPUT: {text}")
#     print(f"{'='*60}\n")

#     # ── Task 1: Conflict Detection ───────────────────────────────────────
#     tokenizer, model = load_model(model_dir, 'conflict')
#     pred, probs = predict_task(text, tokenizer, model)

#     label = LABELS['conflict'][pred]
#     print(f"  1. CONFLICT:  {label}")
#     print(f"     Probabilities: " +
#           "  ".join(f"{LABELS['conflict'][i]}={p:.3f}" for i, p in enumerate(probs)))

#     # If no conflict detected, skip sub-tasks
#     if pred == 0:
#         print(f"\n  No conflict detected. Skipping severity/type/target.")
#         print(f"{'='*60}\n")
#         return {
#             'conflict': {'label': label, 'probs': probs},
#             'severity': None,
#             'type': None,
#             'target': None,
#         }

#     results = {'conflict': {'label': label, 'probs': probs}}

#     # ── Tasks 2-4: Severity, Type, Target ────────────────────────────────
#     for task in ['severity', 'type', 'target']:
#         tokenizer, model = load_model(model_dir, task)
#         pred, probs = predict_task(text, tokenizer, model)
#         label = LABELS[task][pred]

#         task_num = TASKS.index(task) + 1
#         print(f"\n  {task_num}. {task.upper()}:  {label}")
#         print(f"     Probabilities: " +
#               "  ".join(f"{LABELS[task][i]}={p:.3f}" for i, p in enumerate(probs)))

#         results[task] = {'label': label, 'probs': probs}

#     print(f"\n{'='*60}")

#     # ── Summary ──────────────────────────────────────────────────────────
#     print(f"\n  SUMMARY")
#     print(f"  {'─'*40}")
#     print(f"  Conflict : {results['conflict']['label']}")
#     print(f"  Severity : {results['severity']['label']}")
#     print(f"  Type     : {results['type']['label']}")
#     print(f"  Target   : {results['target']['label']}")
#     print(f"{'='*60}\n")

#     return results


# if __name__ == '__main__':
#     parser = argparse.ArgumentParser(description='MalayalamCyberCon Inference')
#     parser.add_argument('--text', type=str, required=True,
#                         help='Input text (Manglish comment thread)')
#     parser.add_argument('--model_dir', type=str, default='./xlmr_large',
#                         help='Path to model directory (default: ./xlmr_large)')
#     args = parser.parse_args()

#     run_pipeline(args.text, args.model_dir)